# 02 · Calidad de datos OECD (bronze)

- **Rol:** maestra de productividad (`PIB_HORA`); contraste de fuerza laboral, fecundidad/nacimientos y dependencia de vejez.
- **Variables:** PIB por hora; LF, EMP, LF_WAP, EMP_WAP, UNE_LF (total y por sexo); TFR y nacidos vivos; dependencia de vejez.
- **Archivos:** `data/bronze/oecd/<fecha más reciente>/{productividad, fuerza_laboral, fecundidad_nacimientos, dependencia_vejez}.csv` (SDMX `csvfilewithlabels`).
Metas del documento: completitud ≥ 95 % nacional, 0 duplicados sobre la clave, valores dentro de rango.

In [1]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from src.transform import lectura_bronze as lb
from src.quality import perfilamiento as pf
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

ANIOS = range(2000, 2026)                      # periodo histórico del documento
PAISES = ["KOR", "JPN", "USA", "DEU", "ITA", "ESP", "FRA"]
prod, fl, fe, dep = (lb.oecd(d) for d in ["productividad", "fuerza_laboral", "fecundidad_nacimientos", "dependencia_vejez"])
print("Archivo bronze:", lb.archivo_reciente("oecd", "productividad.csv").relative_to(lb.RAIZ).as_posix())
pd.Series({"productividad": len(prod), "fuerza_laboral": len(fl),
           "fecundidad_nacimientos": len(fe), "dependencia_vejez": len(dep)}, name="filas")

Archivo bronze: data/bronze/oecd/2026-10-01/productividad.csv


productividad               26
fuerza_laboral            4644
fecundidad_nacimientos    4792
dependencia_vejez          546
Name: filas, dtype: int64

## 1. PIB_HORA (maestra)
Debe ser PIB por hora trabajada, USD PPA, volumen encadenado, base 2020. Esta serie (`DF_PDB 2.0`) reemplazó a la versión archivada `DF_PDB_LV`, que para Corea solo cubría 2011-2023.

In [2]:
display(prod[["MEASURE", "UNIT_MEASURE", "Unit of measure", "PRICE_BASE", "Price base", "BASE_PER", "UNIT_MULT", "OBS_STATUS"]].drop_duplicates())
pib = prod[["REF_AREA", "anio", "valor"]].sort_values("anio")
crec = pib["valor"].pct_change() * 100
comp_pib = pf.completitud(pib, ["REF_AREA"], ANIOS)
print(f"Completitud 2000-2025: {comp_pib.completitud_pct.iloc[0]} % ({comp_pib.anios_con_dato.iloc[0]}/26 años)")
print(f"Valores <= 0: {(pib.valor <= 0).sum()} | nulos: {pib.valor.isna().sum()} | duplicados (REF_AREA, anio): {pf.duplicados(pib, ['REF_AREA', 'anio'])}")
print(f"Rango: {pib.valor.min():.1f} - {pib.valor.max():.1f} USD/h | variación anual: {crec.min():.1f} % a {crec.max():.1f} %")

,MEASURE,UNIT_MEASURE,Unit of measure,PRICE_BASE,Price base,BASE_PER,UNIT_MULT,OBS_STATUS
0,GDPHRS,USD_PPP_H,"US dollars per hour, PPP converted",LR,Chain linked volume (rebased),2020,0,A


Completitud 2000-2025: 100.0 % (26/26 años)
Valores <= 0: 0 | nulos: 0 | duplicados (REF_AREA, anio): 0
Rango: 19.5 - 53.4 USD/h | variación anual: 0.3 % a 9.3 %


## 2. Riesgo de mezclar series si no se filtra
Cada archivo de contraste trae varias series por medida (unidades, sexo, edad, sector, índices). Sin filtrar, una agregación por `anio` las promedia.

**Filtros aplicados** (los de `config.yaml`, más `WORKER_STATUS=_Z` y `ACTIVITY=_Z` en fuerza laboral, que no estaban documentados y son necesarios: `EMP` en `PS` trae además autoempleo y tres sectores).
Los niveles (`PS`) se convierten con `valor × 10^UNIT_MULT`. Para fecundidad solo se usa el nivel país (`CTRY`); los 17 si-do (`TL3`) quedan en bronze sin uso, porque el documento usa las fuentes internacionales solo a nivel país.

In [3]:
DIMS = {"fuerza_laboral": (fl, ["MEASURE", "UNIT_MEASURE", "TRANSFORMATION", "SEX", "AGE", "WORKER_STATUS", "ACTIVITY"]),
        "fecundidad_nacimientos": (fe[fe.TERRITORIAL_LEVEL == "CTRY"], ["MEASURE", "AGE", "SEX", "UNIT_MEASURE"]),
        "dependencia_vejez": (dep, ["MEASURE", "UNIT_MEASURE", "SEX", "AGE"])}
filas = []
for nombre, (df, dims) in DIMS.items():
    for m, g in df[df.REF_AREA == "KOR"].groupby("MEASURE"):
        filas.append({"dataset": nombre, "medida": m, "series_KOR": len(g[dims].drop_duplicates()),
                      "UNIT_MEASURE": ", ".join(sorted(g.UNIT_MEASURE.unique())),
                      "UNIT_MULT": ", ".join(map(str, sorted(g.UNIT_MULT.unique())))})
display(pd.DataFrame(filas))

tfr_mal = fe[(fe.REF_AREA == "KOR") & (fe.MEASURE == "FERT_RATIO") & (fe.anio == 2000)].valor.mean()
tfr_ok = fe[(fe.REF_AREA == "KOR") & (fe.MEASURE == "FERT_RATIO") & (fe.AGE == "_T") & (fe.anio == 2000)].valor.iloc[0]
print(f"Ejemplo TFR KOR 2000 -> sin filtrar AGE: {tfr_mal:.2f} | con AGE=_T: {tfr_ok:.2f}")
print("¿LF_WAP (tasa de participación) presente?:", "LF_WAP" in fl.MEASURE.unique())
print("Áreas por nivel territorial en fecundidad:", fe.groupby("TERRITORIAL_LEVEL").REF_AREA.nunique().to_dict())

,dataset,medida,series_KOR,UNIT_MEASURE,UNIT_MULT
0,fuerza_laboral,EMP,17,"IX, PS, PT_EMP, PT_WAP_SUB","0, 3"
1,fuerza_laboral,EMP_WAP,2,PT_WAP_SUB,0
2,fuerza_laboral,LF,4,"IX, PS","0, 3"
3,fuerza_laboral,UNE_LF,3,PT_LF_SUB,0
4,fecundidad_nacimientos,FERT_RATIO,7,BR_L_W,0
5,fecundidad_nacimientos,LIVE_BIRTHS,2,"10P3HB, BR",0
6,dependencia_vejez,DEPEND_RATIO_OLD,3,PT_POP_Y20T64,0


Ejemplo TFR KOR 2000 -> sin filtrar AGE: 0.42 | con AGE=_T: 1.48
¿LF_WAP (tasa de participación) presente?: False
Áreas por nivel territorial en fecundidad: {'CTRY': 7, 'TL3': 17}


In [4]:
a_personas = lambda d: d.assign(valor=d.valor * 10.0 ** d.UNIT_MULT)
lab = fl[(fl.AGE == "Y_GE15") & (fl.WORKER_STATUS == "_Z") & (fl.ACTIVITY == "_Z") & (fl.UNIT_MEASURE != "IX")]
ctry = fe[fe.TERRITORIAL_LEVEL == "CTRY"]
VARS = {  # variable: (subconjunto filtrado, (min, max) válido)
    "POB_ACTIVA (LF)":            (a_personas(lab[lab.MEASURE == "LF"]), (0, None)),
    "OCUPADOS (EMP)":             (a_personas(lab[lab.MEASURE == "EMP"]), (0, None)),
    "TASA_PARTICIPACION (LF_WAP)": (lab[lab.MEASURE == "LF_WAP"], (0, 100)),
    "TASA_EMPLEO (EMP_WAP)":      (lab[lab.MEASURE == "EMP_WAP"], (0, 100)),
    "TASA_DESEMPLEO (UNE_LF)":    (lab[lab.MEASURE == "UNE_LF"], (0, 100)),
    "TFR (FERT_RATIO, AGE=_T)":   (ctry[(ctry.MEASURE == "FERT_RATIO") & (ctry.AGE == "_T")], (0, 10)),
    "NACIMIENTOS (LIVE_BIRTHS, BR)": (ctry[(ctry.MEASURE == "LIVE_BIRTHS") & (ctry.UNIT_MEASURE == "BR")], (0, None)),
    "DEPENDENCIA_VEJEZ (65+/20-64)": (dep, (0, 100)),
}
CLAVE = ["REF_AREA", "SEX", "anio"]

# Completitud 2000-2025 por país (mínimo entre los sexos disponibles)
comp = {v: pf.completitud(d, ["REF_AREA", "SEX"], ANIOS).groupby("REF_AREA").completitud_pct.min()
        for v, (d, _) in VARS.items() if len(d)}
tabla_comp = pd.DataFrame(comp).T.reindex(columns=PAISES)
tabla_comp["sexos"] = [", ".join(sorted(VARS[v][0].SEX.unique())) for v in tabla_comp.index]
tabla_comp

REF_AREA,KOR,JPN,USA,DEU,ITA,ESP,FRA,sexos
POB_ACTIVA (LF),100.0,100.0,96.2,100.0,100.0,100.0,100.0,"F, M, _T"
OCUPADOS (EMP),100.0,100.0,96.2,100.0,100.0,100.0,100.0,"F, M, _T"
TASA_EMPLEO (EMP_WAP),100.0,100.0,96.2,100.0,100.0,100.0,100.0,"F, M"
TASA_DESEMPLEO (UNE_LF),100.0,100.0,96.2,100.0,100.0,100.0,88.5,"F, M, _T"
"TFR (FERT_RATIO, AGE=_T)",96.2,96.2,88.5,96.2,96.2,96.2,96.2,F
"NACIMIENTOS (LIVE_BIRTHS, BR)",96.2,96.2,100.0,96.2,96.2,96.2,96.2,_T
DEPENDENCIA_VEJEZ (65+/20-64),100.0,100.0,100.0,100.0,100.0,100.0,100.0,"F, M, _T"


## 3. Validez: rangos, unidades y estado de la observación

In [5]:
filas = []
for v, (d, (lo, hi)) in VARS.items():
    k = d[d.REF_AREA == "KOR"]
    filas.append({"variable": v, "filas": len(d), "fuera_rango": len(pf.fuera_de_rango(d, lo, hi)), "nulos": int(d.valor.isna().sum()),
                  "min": d.valor.min(), "max": d.valor.max(),
                  "OBS_STATUS_KOR": k.OBS_STATUS.value_counts().to_dict() if len(k) else "-"})
display(pd.DataFrame(filas).set_index("variable").round(2))

lf_kor = VARS["POB_ACTIVA (LF)"][0].query("REF_AREA == 'KOR' and SEX == '_T' and anio == 2025").valor.iloc[0]
print(f"Unidad niveles: UNIT_MULT=3 (miles). LF KOR 2025 = {lf_kor/1e3:,.0f} miles = {lf_kor/1e6:.1f} M personas (plausible).")
print("Estados no normales en todo fuerza_laboral KOR:", fl[fl.REF_AREA == "KOR"].groupby(["OBS_STATUS", "ACTIVITY"]).size().drop("A", level=0).to_dict())

,filas,fuera_rango,nulos,min,max,OBS_STATUS_KOR
variable,,,,,,
POB_ACTIVA (LF),543,0,0,7071850.00,1.681062e+08,{'A': 78}
OCUPADOS (EMP),543,0,0,5684775.00,1.613455e+08,{'A': 78}
TASA_PARTICIPACION (LF_WAP),0,0,0,NaN,NaN,-
TASA_EMPLEO (EMP_WAP),362,0,0,36.66,6.334000e+01,{'A': 52}
TASA_DESEMPLEO (UNE_LF),537,0,0,2.17,2.667000e+01,{'A': 78}
"TFR (FERT_RATIO, AGE=_T)",173,0,0,0.72,2.120000e+00,{'A': 25}
"NACIMIENTOS (LIVE_BIRTHS, BR)",176,0,0,230028.00,4.316233e+06,{'A': 25}
DEPENDENCIA_VEJEZ (65+/20-64),546,0,0,8.59,6.125000e+01,{'A': 78}


Unidad niveles: UNIT_MULT=3 (miles). LF KOR 2025 = 29,599 miles = 29.6 M personas (plausible).
Estados no normales en todo fuerza_laboral KOR: {('E', 'BTE'): 27, ('E', 'FTT'): 27, ('E', '_Z'): 104}


## 4. Unicidad
Duplicados sobre la clave `(REF_AREA, SEX, anio)` antes y después de aplicar los filtros.

In [6]:
crudo = {"fuerza_laboral": fl, "fecundidad_nacimientos": fe, "dependencia_vejez": dep}
antes = {n: pf.duplicados(d, ["REF_AREA", "MEASURE", "SEX", "anio"]) for n, d in crudo.items()}
print("Duplicados por (REF_AREA, MEASURE, SEX, anio) SIN filtrar:", antes)
dups = {v: pf.duplicados(d, CLAVE) for v, (d, _) in VARS.items()}
pd.Series(dups, name="duplicados_tras_filtro")

Duplicados por (REF_AREA, MEASURE, SEX, anio) SIN filtrar: {'fuerza_laboral': 2658, 'fecundidad_nacimientos': 3617, 'dependencia_vejez': 0}


POB_ACTIVA (LF)                  0
OCUPADOS (EMP)                   0
TASA_PARTICIPACION (LF_WAP)      0
TASA_EMPLEO (EMP_WAP)            0
TASA_DESEMPLEO (UNE_LF)          0
TFR (FERT_RATIO, AGE=_T)         0
NACIMIENTOS (LIVE_BIRTHS, BR)    0
DEPENDENCIA_VEJEZ (65+/20-64)    0
Name: duplicados_tras_filtro, dtype: int64

## 5. Consistencia interna
- Hombres + Mujeres ≈ Total en niveles; `EMP ≤ LF`; `UNE_LF ≈ (LF − EMP) / LF × 100`.
- TFR ≈ Σ tasas específicas por edad (`BR_L_W` ya expresa el aporte por mujer de cada grupo quinquenal); para KOR solo hay 2000-2021 y falta el grupo 40-44.
- Como falta `LF_WAP`, se prueba derivarla: `LF/WAP = EMP_WAP / (1 − UNE_LF/100)` (solo H y M, porque `EMP_WAP` no trae total).

In [7]:
niv = pd.concat([VARS["POB_ACTIVA (LF)"][0], VARS["OCUPADOS (EMP)"][0]]).pivot_table(index=["REF_AREA", "MEASURE", "anio"], columns="SEX", values="valor")
dif_sexo = ((niv.M + niv.F) / niv["_T"] * 100 - 100).abs()
w = pd.concat([VARS["POB_ACTIVA (LF)"][0], VARS["OCUPADOS (EMP)"][0], VARS["TASA_DESEMPLEO (UNE_LF)"][0]]).pivot_table(index=CLAVE, columns="MEASURE", values="valor")
dif_une = ((w.LF - w.EMP) / w.LF * 100 - w.UNE_LF).abs()
print(f"H+M vs Total (LF, EMP): dif máx {dif_sexo.max():.3f} % | casos EMP > LF: {(w.EMP > w.LF).sum()}")
print("UNE_LF vs (LF-EMP)/LF, dif máx en p.p. por país:", dif_une.groupby("REF_AREA").max().round(3).to_dict())

asfr = ctry[(ctry.MEASURE == "FERT_RATIO") & (ctry.AGE != "_T") & (ctry.REF_AREA == "KOR")].groupby("anio").valor.sum()   # BR_L_W: aporte por mujer de cada grupo quinquenal (ya multiplicado por 5)
tfr_k = VARS["TFR (FERT_RATIO, AGE=_T)"][0].query("REF_AREA == 'KOR'").set_index("anio").valor
dif_tfr = pf.dif_pct(tfr_k, asfr).dropna()
print(f"TFR KOR vs ΣASFR ({dif_tfr.index.min()}-{dif_tfr.index.max()}): dif {dif_tfr.min():.1f} % a {dif_tfr.max():.1f} % | grupos de edad KOR: {sorted(ctry[(ctry.REF_AREA == 'KOR') & (ctry.MEASURE == 'FERT_RATIO') & (ctry.AGE != '_T')].AGE.unique())}")

ew = VARS["TASA_EMPLEO (EMP_WAP)"][0].set_index(CLAVE).valor
part = (ew / (1 - w.UNE_LF.reindex(ew.index) / 100)).rename("LF_WAP_derivada")
print("Tasa de participación derivada KOR 2025:", part.xs(("KOR", 2025), level=["REF_AREA", "anio"]).round(1).to_dict())

H+M vs Total (LF, EMP): dif máx 0.021 % | casos EMP > LF: 0
UNE_LF vs (LF-EMP)/LF, dif máx en p.p. por país: {'DEU': 1.032, 'ESP': 0.015, 'FRA': 0.392, 'ITA': 0.017, 'JPN': 0.035, 'KOR': 0.026, 'USA': 0.037}


TFR KOR vs ΣASFR (2000-2021): dif -5.5 % a -0.7 % | grupos de edad KOR: ['Y15T19', 'Y20T24', 'Y25T29', 'Y30T34', 'Y35T39', 'Y45T49']
Tasa de participación derivada KOR 2025: {'F': 45.9, 'M': 57.0}


## 6. Actualidad: último año con dato

In [8]:
ultimo = pd.DataFrame({v: d[d.valor.notna()].groupby("REF_AREA").anio.max() for v, (d, _) in VARS.items() if len(d)}).T.reindex(columns=PAISES)
print(f"PIB_HORA KOR: último año {pib.anio.max()}")
ultimo

PIB_HORA KOR: último año 2025


REF_AREA,KOR,JPN,USA,DEU,ITA,ESP,FRA
POB_ACTIVA (LF),2025,2025,2024,2025,2025,2025,2025
OCUPADOS (EMP),2025,2025,2024,2025,2025,2025,2025
TASA_EMPLEO (EMP_WAP),2025,2025,2024,2025,2025,2025,2025
TASA_DESEMPLEO (UNE_LF),2025,2025,2024,2025,2025,2025,2025
"TFR (FERT_RATIO, AGE=_T)",2024,2024,2023,2024,2024,2024,2024
"NACIMIENTOS (LIVE_BIRTHS, BR)",2024,2024,2025,2024,2024,2024,2024
DEPENDENCIA_VEJEZ (65+/20-64),2025,2025,2025,2025,2025,2025,2025


## 7. Resumen de calidad (Corea, nacional)

In [9]:
def fila(v, d, rol, consistencia, estado=None):
    k = d[d.REF_AREA == "KOR"] if len(d) else d
    compl = pf.completitud(k, ["SEX"], ANIOS).completitud_pct.min() if len(k) else 0.0
    rng = VARS.get(v, (None, (0, None)))[1]
    fr = len(pf.fuera_de_rango(k, *rng)) if len(k) else 0
    dup = pf.duplicados(k, CLAVE) if len(k) else 0
    if estado is None:
        estado = "Bloqueante" if compl == 0 else ("Observación" if compl < 95 or fr or dup else "OK")
    return {"variable": v, "rol": rol, "periodo_KOR": f"{k.anio.min()}-{k.anio.max()}" if len(k) else "sin datos",
            "completitud_KOR_%": compl, "fuera_rango": fr, "duplicados": dup, "consistencia": consistencia, "estado": estado}

pib_k = pib.assign(SEX="_T")
resumen = pd.DataFrame([
    fila("PIB_HORA", pib_k, "maestra", f"USD PPA, LR base 2020; var. anual {crec.min():.1f}% a {crec.max():.1f}%"),
    fila("POB_ACTIVA (LF)", VARS["POB_ACTIVA (LF)"][0], "contraste", f"H+M=T (dif ≤{dif_sexo.max():.2f}%); viene en miles", "Observación"),
    fila("OCUPADOS (EMP)", VARS["OCUPADOS (EMP)"][0], "contraste", "EMP ≤ LF; filtrar WORKER_STATUS/ACTIVITY=_Z; en miles", "Observación"),
    fila("TASA_PARTICIPACION (LF_WAP)", VARS["TASA_PARTICIPACION (LF_WAP)"][0], "contraste", "medida no publicada en DF_SUMTAB"),
    fila("TASA_EMPLEO (EMP_WAP)", VARS["TASA_EMPLEO (EMP_WAP)"][0], "contraste", "solo H y M (sin total)", "Observación"),
    fila("TASA_DESEMPLEO (UNE_LF)", VARS["TASA_DESEMPLEO (UNE_LF)"][0], "contraste", f"≈(LF-EMP)/LF (dif KOR ≤{dif_une.xs('KOR').max():.2f} p.p.)"),
    fila("TFR (FERT_RATIO, AGE=_T)", VARS["TFR (FERT_RATIO, AGE=_T)"][0], "contraste", f"≈ΣASFR (dif {dif_tfr.min():.1f}% a {dif_tfr.max():.1f}%; falta 40-44)"),
    fila("NACIMIENTOS (LIVE_BIRTHS, BR)", VARS["NACIMIENTOS (LIVE_BIRTHS, BR)"][0], "contraste", "filtrar UNIT_MEASURE=BR"),
    fila("DEPENDENCIA_VEJEZ (65+/20-64)", VARS["DEPENDENCIA_VEJEZ (65+/20-64)"][0], "contraste", "definición 65+/20-64 ≠ 65+/15-64", "Observación"),
])
resumen

,variable,rol,periodo_KOR,completitud_KOR_%,fuera_rango,duplicados,consistencia,estado
0,PIB_HORA,maestra,2000-2025,100.0,0,0,"USD PPA, LR base 2020; var. anual 0.3% a 9.3%",OK
1,POB_ACTIVA (LF),contraste,2000-2025,100.0,0,0,H+M=T (dif ≤0.02%); viene en miles,Observación
2,OCUPADOS (EMP),contraste,2000-2025,100.0,0,0,EMP ≤ LF; filtrar WORKER_STATUS/ACTIVITY=_Z; e...,Observación
3,TASA_PARTICIPACION (LF_WAP),contraste,sin datos,0.0,0,0,medida no publicada en DF_SUMTAB,Bloqueante
4,TASA_EMPLEO (EMP_WAP),contraste,2000-2025,100.0,0,0,solo H y M (sin total),Observación
5,TASA_DESEMPLEO (UNE_LF),contraste,2000-2025,100.0,0,0,≈(LF-EMP)/LF (dif KOR ≤0.03 p.p.),OK
6,"TFR (FERT_RATIO, AGE=_T)",contraste,2000-2024,96.2,0,0,≈ΣASFR (dif -5.5% a -0.7%; falta 40-44),OK
7,"NACIMIENTOS (LIVE_BIRTHS, BR)",contraste,2000-2024,96.2,0,0,filtrar UNIT_MEASURE=BR,OK
8,DEPENDENCIA_VEJEZ (65+/20-64),contraste,2000-2025,100.0,0,0,definición 65+/20-64 ≠ 65+/15-64,Observación


## 8. Hallazgos y acciones para silver

**Hallazgos**
- **PIB_HORA (maestra) apto:** 26/26 años 2000-2025 (100 % ≥ 95 %), sin nulos, sin duplicados, todo `OBS_STATUS=A`; unidad USD PPA por hora, volumen encadenado (`LR`), base 2020, 19.5 → 53.4 USD/h con crecimiento anual de 0.3 % a 9.3 %.
- **Falta la tasa de participación OCDE (`LF_WAP`)**: se pidió en la consulta, pero el dataflow `DF_SUMTAB` no la publica (solo trae EMP, EMP_WAP, LF, UNE y UNE_LF). Es bloqueante solo como contraste OCDE de `TASA_PARTICIPACION`; World Bank `SL.TLF.CACT.ZS` cubre ese contraste.
- **Los niveles vienen en miles (`UNIT_MULT=3`)**, no en personas como dice `config.yaml`. LF KOR 2025 = 29 599 miles.
- **Mezclar series es un riesgo real**: sin filtrar hay 2 658 duplicados de clave en fuerza laboral y 3 617 en fecundidad. Promediar TFR sin `AGE=_T` da 0.42 en vez de 1.48 (KOR 2000). Tras los filtros quedan 0 duplicados y 0 valores fuera de rango.
- `EMP` en `PS` incluye autoempleo (`WORKER_STATUS`) y 3 sectores (`ACTIVITY`), además de un índice (`IX`). Los estados `E` (estimado) de KOR están solo en esas series; las series filtradas son 100 % `A`.
- `EMP_WAP` no trae total (solo H y M). Las tasas usan `TRANSFORMATION=G1` ("tasa de crecimiento") aunque son niveles en %, así que no se puede filtrar por `TRANSFORMATION=_Z`.
- Completitud KOR: fuerza laboral y dependencia 100 %; TFR y nacimientos 96.2 % (hasta 2024). Entre los países, la más baja es USA en TFR (88.5 %, 2001-2023) y la de FRA en desempleo por sexo (88.5 %, desde 2003).
- Consistencia: H+M = Total (dif ≤ 0.02 %), EMP ≤ LF siempre, UNE_LF ≈ (LF−EMP)/LF (KOR ≤ 0.03 p.p.). ΣASFR queda entre 0.7 % y 5.5 % por debajo de la TFR porque a KOR le falta el grupo 40-44 (la brecha crece con el retraso de la maternidad).
- Dependencia de vejez OCDE = 65+/20-64: no es comparable 1 a 1 con nuestra definición (65+/15-64).

**Acciones para silver**
- Aplicar los filtros: `AGE=Y_GE15`, `WORKER_STATUS=_Z`, `ACTIVITY=_Z`, `UNIT_MEASURE≠IX` (fuerza laboral); `TERRITORIAL_LEVEL=CTRY` + `FERT_RATIO/AGE=_T` y `LIVE_BIRTHS/BR` (fecundidad). Luego validar la unicidad de `(REF_AREA, MEASURE, SEX, anio)` y mandar a `ctl.rechazos` lo que sobre.
- Multiplicar los niveles por `10^UNIT_MULT` (miles → personas) y corregir la nota de `config.yaml` (los `PS` vienen en miles).
- Contraste de `TASA_PARTICIPACION`: usar World Bank, o derivar el dato OCDE por sexo como `EMP_WAP / (1 − UNE_LF/100)` (KOR 2025: H 57.0 %, M 45.9 %), marcado como derivado.
- En `silver.conciliacion`, comparar la dependencia OCDE contra 65+/20-64 recalculada desde KOSIS, o dejar los pares fuera de la meta de ±3 % con el motivo "definición distinta".
- Dejar fuera de silver el nivel `TL3` (17 si-do) y las tasas específicas por edad. Guardar `OBS_STATUS` como metadato (`B` = quiebre de serie; aparece en todos los países de comparación, no en KOR).